<a href="https://colab.research.google.com/github/baonguyen31/myTodoList/blob/main/PhobertModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets seqeval underthesea -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 64.1 MB/s eta 0:00:00


Để tải lên tệp JSON của bạn, bạn sẽ cần sử dụng tính năng tải lên tệp của Colab. Sau khi chạy ô mã bên dưới, một nút sẽ xuất hiện cho phép bạn duyệt và chọn tệp JSON từ máy tính của mình.

In [ ]:
from google.colab import files
import json

uploaded = files.upload()  # chọn dataset.json

with open("dataset.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

print(f"Tổng số câu: {len(dataset)}")
print(f"\nVí dụ câu đầu:")
print("Tokens:", dataset[0]["tokens"])
print("Labels:", dataset[0]["labels"])

Saving dataset.json to dataset.json
Tổng số câu: 26961

Ví dụ câu đầu:
Tokens: ['Họp', 'với', 'đối_tác', 'lúc', '7', 'giờ', 'chiều', 'thứ', 'ba']
Labels: ['B-TASK', 'O', 'O', 'O', 'B-TIME', 'I-TIME', 'I-TIME', 'B-DATE', 'I-DATE']


In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Không có GPU!")

GPU: Tesla T4


In [ ]:
from transformers import PhobertTokenizerFast

MODEL_NAME = "vinai/phobert-base"
tokenizer = PhobertTokenizerFast.from_pretrained(MODEL_NAME, use_fast=False)

print("Tokenizer loaded!")
print("Ví dụ tokenize:", tokenizer.tokenize("Họp với sếp lúc 9 giờ sáng"))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded!
Ví dụ tokenize: ['Họp', 'với', 'sếp', 'lúc', '9', 'giờ', 'sáng']


In [ ]:
LABELS = ["O", "B-TASK", "I-TASK", "B-TIME", "I-TIME", "B-DATE", "I-DATE"]

label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for i, label in enumerate(LABELS)}

print("label2id:", label2id)

label2id: {'O': 0, 'B-TASK': 1, 'I-TASK': 2, 'B-TIME': 3, 'I-TIME': 4, 'B-DATE': 5, 'I-DATE': 6}


In [ ]:
import torch

def tokenize_and_align(examples, tokenizer, label2id, max_length=128):
    all_input_ids       = []
    all_attention_masks = []
    all_labels          = []

    for item in examples:
        tokens = [t.replace("_", " ") for t in item["tokens"]]
        labels = item["labels"]

        # Tokenize từng word riêng để biết word nào → subwords nào
        all_subwords  = []
        all_label_ids = []

        for word, label in zip(tokens, labels):
            subwords = tokenizer.tokenize(word)
            if len(subwords) == 0:
                subwords = [tokenizer.unk_token]

            all_subwords.append(subwords[0])
            all_label_ids.append(label2id[label])  # subword đầu → label thật

            for sw in subwords[1:]:
                all_subwords.append(sw)
                all_label_ids.append(-100)          # subword sau → bỏ qua

        # Cắt nếu quá dài (trừ 2 chỗ cho [CLS] và [SEP])
        max_len = max_length - 2
        all_subwords  = all_subwords[:max_len]
        all_label_ids = all_label_ids[:max_len]

        # Chuyển subwords → ids và thêm special tokens
        input_ids = (
            [tokenizer.cls_token_id] +
            tokenizer.convert_tokens_to_ids(all_subwords) +
            [tokenizer.sep_token_id]
        )
        label_ids_final = [-100] + all_label_ids + [-100]

        # Padding
        attention_mask = [1] * len(input_ids)
        padding_len = max_length - len(input_ids)

        input_ids      += [tokenizer.pad_token_id] * padding_len
        attention_mask += [0] * padding_len
        label_ids_final += [-100] * padding_len

        all_input_ids.append(torch.tensor(input_ids))
        all_attention_masks.append(torch.tensor(attention_mask))
        all_labels.append(torch.tensor(label_ids_final))

    return all_input_ids, all_attention_masks, all_labels


# Chạy
input_ids, attention_masks, label_ids = tokenize_and_align(
    dataset, tokenizer, label2id
)

print(f"Đã tokenize {len(input_ids)} câu")
print(f"\nVí dụ câu đầu (15 token đầu):")
print(f"{'Subword':25} {'Label'}")
print("-" * 35)

tokens_example = tokenizer.convert_ids_to_tokens(input_ids[0].tolist())[:15]
labels_example = label_ids[0].tolist()[:15]

for tok, lab in zip(tokens_example, labels_example):
    lab_str = id2label.get(lab, "IGN") if lab != -100 else "IGN"
    print(f"  {tok:23} {lab_str}")

Đã tokenize 26961 câu

Ví dụ câu đầu (15 token đầu):
Subword                   Label
-----------------------------------
  <s>                     IGN
  Họp                     B-TASK
  với                     O
  đối                     O
  tác                     IGN
  lúc                     O
  7                       B-TIME
  giờ                     I-TIME
  chiều                   I-TIME
  thứ                     B-DATE
  ba                      I-DATE
  </s>                    IGN
  <pad>                   IGN
  <pad>                   IGN
  <pad>                   IGN


In [ ]:
print("Dữ liệu df DataFrame:")
display(df.head())
print("Thông tin df DataFrame:")
df.info()

Dữ liệu df DataFrame:


NameError: name 'df' is not defined

In [ ]:
import pandas as pd
from collections import Counter

print("=" * 50)
print("  KIỂM TRA DATASET TRƯỚC KHI TRAIN")
print("=" * 50)

# ── 1. Tổng quan ──────────────────────────────────────
print(f"\nTổng câu ban đầu: {len(dataset)}")

# ── 2. Kiểm tra token/label lệch ─────────────────────
mismatch = [i for i, d in enumerate(dataset)
            if len(d["tokens"]) != len(d["labels"])]
print(f"Câu bị lệch token/label: {len(mismatch)}")
if mismatch:
    print(f"  → Xóa {len(mismatch)} câu lỗi")
    dataset = [d for i, d in enumerate(dataset) if i not in mismatch]

# ── 3. Kiểm tra label không hợp lệ ───────────────────
VALID_LABELS = set(LABELS)
invalid = []
for i, d in enumerate(dataset):
    for lab in d["labels"]:
        if lab not in VALID_LABELS:
            invalid.append(i)
            break
print(f"Câu có label không hợp lệ: {len(invalid)}")
if invalid:
    dataset = [d for i, d in enumerate(dataset) if i not in invalid]

# ── 4. Kiểm tra BIO consistency ───────────────────────
# I-X không được xuất hiện nếu trước đó không có B-X
bio_errors = []
for i, d in enumerate(dataset):
    prev = "O"
    for lab in d["labels"]:
        if lab.startswith("I-"):
            entity = lab[2:]
            if not (prev == f"B-{entity}" or prev == f"I-{entity}"):
                bio_errors.append(i)
                break
        prev = lab
print(f"Câu vi phạm BIO: {len(bio_errors)}")
if bio_errors:
    dataset = [d for i, d in enumerate(dataset) if i not in bio_errors]

# ── 5. Xóa câu trùng lặp ─────────────────────────────
seen = set()
unique_dataset = []
for d in dataset:
    key = " ".join(d["tokens"])
    if key not in seen:
        seen.add(key)
        unique_dataset.append(d)
duplicates = len(dataset) - len(unique_dataset)
print(f"Câu trùng lặp: {duplicates}")
dataset = unique_dataset

# ── 6. Kiểm tra độ dài ───────────────────────────────
lengths = [len(d["tokens"]) for d in dataset]
too_short = sum(1 for l in lengths if l < 3)
too_long  = sum(1 for l in lengths if l > 50)
print(f"\nĐộ dài câu:")
print(f"  Ngắn nhất:  {min(lengths)} tokens")
print(f"  Dài nhất:   {max(lengths)} tokens")
print(f"  Trung bình: {sum(lengths)/len(lengths):.1f} tokens")
print(f"  Quá ngắn (<3):  {too_short} câu → xóa")
print(f"  Quá dài  (>50): {too_long} câu → giữ, sẽ truncate")
dataset = [d for d in dataset if len(d["tokens"]) >= 3]

# ── 7. Phân phối label ────────────────────────────────
all_labels_flat = [lab for d in dataset for lab in d["labels"]]
label_counts = Counter(all_labels_flat)
total = len(all_labels_flat)

print(f"\nPhân phối label:")
for lab in LABELS:
    count = label_counts.get(lab, 0)
    pct   = count / total * 100
    bar   = "█" * int(pct / 2)
    print(f"  {lab:12} {count:6,}  ({pct:5.1f}%)  {bar}")

# ── 8. Kết quả sau làm sạch ───────────────────────────
print(f"\n{'=' * 50}")
print(f"Dataset sau làm sạch: {len(dataset)} câu")
print(f"{'=' * 50}")

  KIỂM TRA DATASET TRƯỚC KHI TRAIN

Tổng câu ban đầu: 26961
Câu bị lệch token/label: 0
Câu có label không hợp lệ: 0
Câu vi phạm BIO: 405
Câu trùng lặp: 0

Độ dài câu:
  Ngắn nhất:  2 tokens
  Dài nhất:   12 tokens
  Trung bình: 8.2 tokens
  Quá ngắn (<3):  6 câu → xóa
  Quá dài  (>50): 0 câu → giữ, sẽ truncate

Phân phối label:
  O            40,297  ( 18.6%)  █████████
  B-TASK       26,550  ( 12.2%)  ██████
  I-TASK       25,028  ( 11.5%)  █████
  B-TIME       26,391  ( 12.2%)  ██████
  I-TIME       46,612  ( 21.5%)  ██████████
  B-DATE       26,316  ( 12.1%)  ██████
  I-DATE       25,965  ( 12.0%)  █████

Dataset sau làm sạch: 26550 câu


In [ ]:
import pandas as pd

df = pd.DataFrame(dataset)

print("Dữ liệu df DataFrame:")
display(df.head())
print("Thông tin df DataFrame:")
df.info()

Dữ liệu df DataFrame:


,tokens,labels
0,"[Họp, với, đối_tác, lúc, 7, giờ, chiều, thứ, ba]","[B-TASK, O, O, O, B-TIME, I-TIME, I-TIME, B-DA..."
1,"[Ăn, trưa, với, team, lúc, 4, giờ, chiều, ngày...","[B-TASK, I-TASK, O, O, O, B-TIME, I-TIME, I-TI..."
2,"[Ăn, trưa, với, gia_đình, 6, h, sáng, thứ, sáu]","[B-TASK, I-TASK, O, O, B-TIME, I-TIME, I-TIME,..."
3,"[Deploy, ứng_dụng, lúc, 6, giờ, sáng, hôm, kia]","[B-TASK, I-TASK, O, B-TIME, I-TIME, I-TIME, B-..."
4,"[Ăn, trưa, với, bạn_bè, lúc, 7, giờ, sáng, ngà...","[B-TASK, I-TASK, O, O, O, B-TIME, I-TIME, I-TI..."


Thông tin df DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26550 entries, 0 to 26549
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   tokens  26550 non-null  object
 1   labels  26550 non-null  object
dtypes: object(2)
memory usage: 415.0+ KB


In [ ]:
from torch.utils.data import Dataset, DataLoader, random_split

class NERDataset(Dataset):
    def __init__(self, input_ids, attention_masks, labels):
        self.input_ids = input_ids
        self.attention_masks = attention_masks
        self.labels = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_masks[idx],
            "labels":         self.labels[idx],
        }

full_dataset = NERDataset(input_ids, attention_masks, label_ids)

# 80% train, 20% validation
train_size = int(0.8 * len(full_dataset))
val_size   = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16)

print(f"Train: {train_size} câu")
print(f"Val:   {val_size} câu")
print(f"Batch size: 16")
print(f"Số batches train: {len(train_loader)}")

Train: 21568 câu
Val:   5393 câu
Batch size: 16
Số batches train: 1348


In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded trên: {device}")
print(f"Tổng số parameters: {total_params:,}")

config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

RobertaForTokenClassification LOAD REPORT from: vinai/phobert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.weight          | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.bias                 | MISSING    | 
classifier.weight               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded trên: cuda
Tổng số parameters: 134,413,063


In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

EPOCHS    = 5
LR        = 2e-5

optimizer = AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0

    for batch in loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def eval_epoch(model, loader, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            total_loss += outputs.loss.item()

    return total_loss / len(loader)


# ── Train ──────────────────────────────────────────────────────────────────
print(f"Bắt đầu train {EPOCHS} epochs trên {device}...\n")

best_val_loss = float("inf")

for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss   = eval_epoch(model, val_loader, device)

    marker = " ← best" if val_loss < best_val_loss else ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        # lưu model tốt nhất
        model.save_pretrained("/content/phobert-ner-best")
        tokenizer.save_pretrained("/content/phobert-ner-best")

    print(f"Epoch {epoch+1}/{EPOCHS} | train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f}{marker}")

print(f"\nTrain xong! Best val_loss: {best_val_loss:.4f}")
print("Model đã lưu tại /content/phobert-ner-best")

Bắt đầu train 5 epochs trên cuda...



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 1/5 | train_loss: 0.2212 | val_loss: 0.0033 ← best


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 2/5 | train_loss: 0.0034 | val_loss: 0.0013 ← best


...

Epoch 3/5 | train_loss: 0.0014 | val_loss: 0.0006 ← best


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 4/5 | train_loss: 0.0008 | val_loss: 0.0004 ← best


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 5/5 | train_loss: 0.0006 | val_loss: 0.0003 ← best

Train xong! Best val_loss: 0.0003
Model đã lưu tại /content/phobert-ner-best


In [ ]:
# Test với câu HOÀN TOÀN MỚI — không có trong template
test_sentences = [
    "Chiều nay tôi cần nộp hồ sơ trước 5 giờ",
    "Nhớ gọi điện cho bác sĩ vào sáng thứ tư",
    "Deadline dự án là ngày 15 tháng 5",
    "Tối nay ăn tối với gia đình lúc 7 rưỡi",
    "Họp khẩn với ban giám đốc lúc 8h sáng mai",
]

from underthesea import word_tokenize

model.eval()
print("Test với câu mới hoàn toàn:\n")

for sent in test_sentences:
    # tokenize
    tokens = word_tokenize(sent, format="text").split()
    tokens_clean = [t.replace("_", " ") for t in tokens]

    subwords = []
    word_starts = []
    for i, word in enumerate(tokens_clean):
        sws = tokenizer.tokenize(word) or [tokenizer.unk_token]
        word_starts.append(len(subwords))
        subwords.extend(sws)

    input_ids = torch.tensor([[
        tokenizer.cls_token_id,
        *tokenizer.convert_tokens_to_ids(subwords[:126]),
        tokenizer.sep_token_id
    ]]).to(device)

    with torch.no_grad():
        logits = model(input_ids=input_ids).logits
    preds = logits.argmax(dim=-1).squeeze().tolist()

    # chỉ lấy pred của subword đầu mỗi từ (bỏ CLS)
    word_preds = [id2label[preds[pos + 1]] for pos in word_starts
                  if pos + 1 < len(preds)]

    print(f"Câu: {sent}")
    task_parts, time_parts, date_parts = [], [], []
    for tok, lab in zip(tokens, word_preds):
        if "TASK" in lab: task_parts.append(tok)
        elif "TIME" in lab: time_parts.append(tok)
        elif "DATE" in lab: date_parts.append(tok)

    print(f"  TASK: {' '.join(task_parts) or '(không tìm thấy)'}")
    print(f"  TIME: {' '.join(time_parts) or '(không tìm thấy)'}")
    print(f"  DATE: {' '.join(date_parts) or '(không tìm thấy)'}")
    print()

Test với câu mới hoàn toàn:

Câu: Chiều nay tôi cần nộp hồ sơ trước 5 giờ
  TASK: Chiều nộp hồ_sơ
  TIME: 5 giờ
  DATE: (không tìm thấy)

Câu: Nhớ gọi điện cho bác sĩ vào sáng thứ tư
  TASK: Nhớ gọi điện
  TIME: sáng
  DATE: thứ tư

Câu: Deadline dự án là ngày 15 tháng 5
  TASK: Deadline dự_án
  TIME: (không tìm thấy)
  DATE: ngày 15 tháng 5

Câu: Tối nay ăn tối với gia đình lúc 7 rưỡi
  TASK: Tối nay ăn tối
  TIME: 7 rưỡi
  DATE: (không tìm thấy)

Câu: Họp khẩn với ban giám đốc lúc 8h sáng mai
  TASK: Họp khẩn
  TIME: 8 h sáng_mai
  DATE: (không tìm thấy)



In [ ]:
from seqeval.metrics import classification_report

def predict(model, loader, device, id2label):
    model.eval()
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"]

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = outputs.logits.argmax(dim=-1).cpu()

            for pred_seq, label_seq in zip(preds, labels):
                pred_tags  = []
                label_tags = []
                for p, l in zip(pred_seq, label_seq):
                    if l.item() == -100:
                        continue          # bỏ qua padding và subwords phụ
                    pred_tags.append(id2label[p.item()])
                    label_tags.append(id2label[l.item()])
                all_preds.append(pred_tags)
                all_labels.append(label_tags)

    return all_preds, all_labels

preds, labels_true = predict(model, val_loader, device, id2label)
print(classification_report(labels_true, preds))

              precision    recall  f1-score   support

        DATE       1.00      1.00      1.00      5343
        TASK       1.00      1.00      1.00      5393
        TIME       1.00      1.00      1.00      5435

   micro avg       1.00      1.00      1.00     16171
   macro avg       1.00      1.00      1.00     16171
weighted avg       1.00      1.00      1.00     16171



In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/phobert-ner-best", "zip", "/content/phobert-ner-best")
files.download("/content/phobert-ner-best.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>